# CalibrateQwen 04: evaluation and report artifacts
We sample a base model or checkpoint, compute calibration and selective-prediction metrics, and render publication-ready figures.

In [ ]:
from pathlib import Path
REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha
%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
from evaluation.sample_model import SamplingConfig, sample_dataset

RUN_NAME = 'base_qwen35_4b'
CHECKPOINT = None  # Paste a tinker:// checkpoint to evaluate a trained run
LIMIT = 250  # Use None for the full test split
prediction_path = f'results/predictions/{RUN_NAME}_test.jsonl'
sampling_config = SamplingConfig(
    output_path=prediction_path,
    split='test',
    model_name='Qwen/Qwen3.5-4B',
    checkpoint_path=CHECKPOINT,
    confidence_format='numeric',
    score_options=True,
    limit=LIMIT,
)
await sample_dataset(sampling_config)

In [ ]:
from evaluation.evaluate_predictions import evaluate_file
from evaluation.plot_results import plot_metrics

metrics_path = f'results/metrics/{RUN_NAME}_test.json'
metrics = evaluate_file(
    predictions_path=prediction_path,
    output_path=metrics_path,
    confidence_source='verbal',
)
plot_metrics(metrics_path, 'results/figures', RUN_NAME)
{key: metrics[key] for key in ['accuracy', 'ece', 'binary_brier', 'aurc', 'mean_error_confidence', 'format_validity']}

In [ ]:
# Repeat the evaluation with answer-token probabilities.
probability_metrics_path = f'results/metrics/{RUN_NAME}_answer_probability.json'
probability_metrics = evaluate_file(
    predictions_path=prediction_path,
    output_path=probability_metrics_path,
    confidence_source='answer_probability',
)
{key: probability_metrics[key] for key in ['accuracy', 'ece', 'multiclass_brier', 'multiclass_nll', 'aurc']}

In [ ]:
# Fit temperature and the 80 percent coverage threshold on validation predictions.
from evaluation.calibrate import apply_calibration, fit_calibration
from data.common import read_jsonl

validation_path = f'results/predictions/{RUN_NAME}_validation.jsonl'
await sample_dataset(SamplingConfig(
    output_path=validation_path,
    split='validation',
    model_name='Qwen/Qwen3.5-4B',
    checkpoint_path=CHECKPOINT,
    confidence_format='numeric',
    score_options=True,
    limit=LIMIT,
))
calibration = fit_calibration(list(read_jsonl(validation_path)), target_coverage=0.8)
calibration

In [ ]:
# Publish a final sampler checkpoint as a private Hugging Face PEFT adapter.
FINAL_CHECKPOINT = None  # Example: tinker://run-id/sampler_weights/final
MODEL_REPO = 'ritwikraha/calibrate-qwen-student'
if FINAL_CHECKPOINT:
    !python -m tinker.cli checkpoint push-hf {FINAL_CHECKPOINT} --repo {MODEL_REPO}